In [14]:
!pip install groq

In [15]:
!pip install langchain openai dotenv pydantic

In [16]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent 
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from pydantic import BaseModel, Field
from typing import Literal

In [17]:
load_dotenv()

True

In [26]:
model = ChatOpenAI(model="gpt-4o-mini")

In [27]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
model = ChatGroq(
    model="openai/gpt-oss-20b",
)

In [19]:
prompt = """
Understand the coding request:

For any request, identify the following :

- task type
- objective
- files involved 

Return the response in the following JSON format:
{
    "task_type": "type of task",
    "objective": "objective of the task",
    "files_involved": "list of files involved"
}

"""

In [20]:
request = "Identify the bug in the auth.py file where the exchange token is not getting generated"

In [21]:
response = model.invoke(prompt + request)
print(response.content)

```json
{
    "task_type": "bug identification",
    "objective": "Identify the bug in auth.py where the exchange token is not being generated",
    "files_involved": ["auth.py"]
}
```

Could you please share the contents of `auth.py` (or at least the relevant portion where the exchange token is generated) so I can pinpoint the issue?


In [22]:
class CodingRequest(BaseModel):
    """Structured representation of the coding request"""

    task_type: Literal[
        "implement",
        "debug",
        "review",
        "explain"
    ] = Field(description="Type of task to be performed")

    objective: str = Field(description="A concise description of what needs to be achieved")

    need_code_changes: bool = Field(
        description="Whether fulfilling this task required modifying the source code"
    )

    target_files: list[str] = Field(
        description="List of files that are relevant to the task, e.g. ['auth.py', 'models.py']"
    )

In [23]:
structured_output_model = model.with_structured_output(CodingRequest)

In [28]:
result = structured_output_model.invoke("""
    Fix the login bug in auth.py. Expired sessions currently produce HTTP 500.

""")

In [29]:
print(type(result))

<class '__main__.CodingRequest'>


In [ ]:
# CodingRequest.model_json_schema()

{'description': 'Structured representation of the coding request',
 'properties': {'task_type': {'description': 'Type of task to be performed',
   'enum': ['implement', 'debug', 'review', 'explain'],
   'title': 'Task Type',
   'type': 'string'},
  'objective': {'description': 'A concise description of what needs to be achieved',
   'title': 'Objective',
   'type': 'string'},
  'need_code_changes': {'description': 'Whether fulfilling this task required modifying the source code',
   'title': 'Need Code Changes',
   'type': 'boolean'},
  'target_files': {'description': "List of files that are relevant to the task, e.g. ['auth.py', 'models.py']",
   'items': {'type': 'string'},
   'title': 'Target Files',
   'type': 'array'}},
 'required': ['task_type', 'objective', 'need_code_changes', 'target_files'],
 'title': 'CodingRequest',
 'type': 'object'}

In [38]:
# Using Provider strategy
# agent = create_agent(
#     model=model,
#     tools=[],
#     response_format=ProviderStrategy(CodingRequest),
#     system_prompt=prompt
# )
# Using tool Strategy
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(CodingRequest),
    system_prompt=prompt
)


In [40]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Identify the bug in the auth.py file where the exchange token is not getting generated"}
    ]
})
print(type(result["structured_response"]))
print(result["structured_response"])

<class '__main__.CodingRequest'>
task_type='debug' objective='Identify bug in auth.py where exchange token is not generated' need_code_changes=False target_files=['auth.py']


In [41]:
print(result)

{'messages': [HumanMessage(content='Identify the bug in the auth.py file where the exchange token is not getting generated', additional_kwargs={}, response_metadata={}, id='65a7fc84-0e99-4ce9-9b5d-6235105a7de5'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to parse user request: "Identify the bug in the auth.py file where the exchange token is not getting generated". We need to determine task type, objective, files involved. Task type likely "debug". Objective: identify bug in auth.py where exchange token not generated. Files involved: auth.py. So JSON: task_type: "debug", objective: "Identify bug in auth.py where exchange token is not generated", files_involved: ["auth.py"].\n\nWe need to call the function.', 'tool_calls': [{'id': 'fc_06fd9f9a-8754-4d17-a537-202ff5dee7d7', 'function': {'arguments': '{"need_code_changes":false,"objective":"Identify bug in auth.py where exchange token is not generated","target_files":["auth.py"],"task_type":"debug"}', 'name': 

In [42]:
for message in result["messages"]:
    print(message.pretty_print())
    print("===================\n\n")

================================ Human Message =================================

Identify the bug in the auth.py file where the exchange token is not getting generated
None


================================== Ai Message ==================================
Tool Calls:
  CodingRequest (fc_06fd9f9a-8754-4d17-a537-202ff5dee7d7)
 Call ID: fc_06fd9f9a-8754-4d17-a537-202ff5dee7d7
  Args:
    need_code_changes: False
    objective: Identify bug in auth.py where exchange token is not generated
    target_files: ['auth.py']
    task_type: debug
None


================================= Tool Message =================================
Name: CodingRequest

Returning structured response: task_type='debug' objective='Identify bug in auth.py where exchange token is not generated' need_code_changes=False target_files=['auth.py']
None


